# Data Preprocessing, Models

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score, auc, roc_auc_score, confusion_matrix, balanced_accuracy_score,
    roc_curve, accuracy_score
)

In [2]:
import os
# df = pd.read_csv(os.path.join('..', 'data', 'data_clean.csv'), index_col=0).reset_index()
df = pd.read_csv(os.path.join('..', 'data', 'data_clean.csv'), index_col=0)
df

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,stroke
0,Urban,Male,0,1,Yes,Private,formerly smoked,67.0,228.69,36.6,1
2,Rural,Male,0,1,Yes,Private,never smoked,80.0,105.92,32.5,1
3,Urban,Female,0,0,Yes,Private,smokes,49.0,171.23,34.4,1
4,Rural,Female,1,0,Yes,Self-employed,never smoked,79.0,174.12,24.0,1
5,Urban,Male,0,0,Yes,Private,formerly smoked,81.0,186.21,29.0,1
...,...,...,...,...,...,...,...,...,...,...,...
5104,Rural,Female,0,0,No,children,Unknown,13.0,103.08,18.6,0
5106,Urban,Female,0,0,Yes,Self-employed,never smoked,81.0,125.20,40.0,0
5107,Rural,Female,0,0,Yes,Self-employed,never smoked,35.0,82.99,30.6,0
5108,Rural,Male,0,0,Yes,Private,formerly smoked,51.0,166.29,25.6,0


### Feature Engineering and Transformations

In [3]:
df['is_urban'] = (df['Residence_type'] == 'Urban').astype(np.int8)
df['is_male'] = (df['gender'] == 'Male').astype(np.int8)
df['ever_married'] = (df['ever_married'] == 'Yes').astype(np.int8)
df.head()

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,stroke,is_urban,is_male
0,Urban,Male,0,1,1,Private,formerly smoked,67.0,228.69,36.6,1,1,1
2,Rural,Male,0,1,1,Private,never smoked,80.0,105.92,32.5,1,0,1
3,Urban,Female,0,0,1,Private,smokes,49.0,171.23,34.4,1,1,0
4,Rural,Female,1,0,1,Self-employed,never smoked,79.0,174.12,24.0,1,0,0
5,Urban,Male,0,0,1,Private,formerly smoked,81.0,186.21,29.0,1,1,1


In [4]:
# df = df[[
#     'age', 'is_male', 'is_urban', 'ever_married', 'work_type', 'smoking_status',
#     'bmi', 'hypertension', 'heart_disease', 'avg_glucose_level', 'stroke'
# ]]
# df.head()

In [5]:
df.dtypes

Residence_type        object
gender                object
hypertension           int64
heart_disease          int64
ever_married            int8
work_type             object
smoking_status        object
age                  float64
avg_glucose_level    float64
bmi                  float64
stroke                 int64
is_urban                int8
is_male                 int8
dtype: object

Things like age groups and bmi categories can help the model pick up on non-linear patterns that raw continuous values sometimes miss.

In [6]:
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 18, 35, 50, 65, np.inf],
    labels=['<18', '18-35', '36-50', '51-65', '65+']
)

df['bmi_category'] = pd.cut(
    df['bmi'],
    bins=[0, 18.5, 25, 30, np.inf],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese']
)

df['avg_glucose_category'] = pd.cut(
    df['avg_glucose_level'],
    bins=[0, 70, 100, 250, np.inf],
    labels=['Low', 'Normal', 'High', 'VeryHigh']
)

In [7]:
df['ever_smoked'] = (df['smoking_status'].apply(
    lambda x: 1 if x in ['formerly smoked', 'smokes'] else 0
)).astype(np.int8)

df['age_bmi_interaction'] = df['age'] * df['bmi']

df['health_risk_score'] = (
    df['hypertension'] + df['heart_disease'] + df['ever_smoked'] + df['age'] / 100
)

df['work_residence_interaction'] = df['work_type'] + "_" + df['Residence_type']

df['marital_age_interaction'] = df['ever_married'] * df['age']

df.sample(3)

,Residence_type,gender,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,...,is_urban,is_male,age_group,bmi_category,avg_glucose_category,ever_smoked,age_bmi_interaction,health_risk_score,work_residence_interaction,marital_age_interaction
4844,Urban,Female,0,0,0,children,Unknown,4.0,72.49,16.9,...,1,0,<18,Underweight,Normal,0,67.6,0.04,children_Urban,0.0
3648,Urban,Female,0,0,0,Private,never smoked,22.0,68.40,37.5,...,1,0,18-35,Obese,Low,0,825.0,0.22,Private_Urban,0.0
704,Rural,Female,0,0,0,Private,Unknown,19.0,91.69,39.5,...,0,0,18-35,Obese,Normal,0,750.5,0.19,Private_Rural,0.0


In [8]:
df.dtypes

Residence_type                  object
gender                          object
hypertension                     int64
heart_disease                    int64
ever_married                      int8
work_type                       object
smoking_status                  object
age                            float64
avg_glucose_level              float64
bmi                            float64
stroke                           int64
is_urban                          int8
is_male                           int8
age_group                     category
bmi_category                  category
avg_glucose_category          category
ever_smoked                       int8
age_bmi_interaction            float64
health_risk_score              float64
work_residence_interaction      object
marital_age_interaction        float64
dtype: object

In [9]:
df.drop(['Residence_type', 'gender'], axis=1, inplace=True)
df

,hypertension,heart_disease,ever_married,work_type,smoking_status,age,avg_glucose_level,bmi,stroke,is_urban,is_male,age_group,bmi_category,avg_glucose_category,ever_smoked,age_bmi_interaction,health_risk_score,work_residence_interaction,marital_age_interaction
0,0,1,1,Private,formerly smoked,67.0,228.69,36.6,1,1,1,65+,Obese,High,1,2452.2,2.67,Private_Urban,67.0
2,0,1,1,Private,never smoked,80.0,105.92,32.5,1,0,1,65+,Obese,High,0,2600.0,1.80,Private_Rural,80.0
3,0,0,1,Private,smokes,49.0,171.23,34.4,1,1,0,36-50,Obese,High,1,1685.6,1.49,Private_Urban,49.0
4,1,0,1,Self-employed,never smoked,79.0,174.12,24.0,1,0,0,65+,Normal,High,0,1896.0,1.79,Self-employed_Rural,79.0
5,0,0,1,Private,formerly smoked,81.0,186.21,29.0,1,1,1,65+,Overweight,High,1,2349.0,1.81,Private_Urban,81.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5104,0,0,0,children,Unknown,13.0,103.08,18.6,0,0,0,<18,Normal,High,0,241.8,0.13,children_Rural,0.0
5106,0,0,1,Self-employed,never smoked,81.0,125.20,40.0,0,1,0,65+,Obese,High,0,3240.0,0.81,Self-employed_Urban,81.0
5107,0,0,1,Self-employed,never smoked,35.0,82.99,30.6,0,0,0,18-35,Obese,Normal,0,1071.0,0.35,Self-employed_Rural,35.0
5108,0,0,1,Private,formerly smoked,51.0,166.29,25.6,0,0,1,51-65,Overweight,High,1,1305.6,1.51,Private_Rural,51.0


### Feature Encoding, Scaling

In [10]:
cat_encode_cols = [
    'hypertension', 'heart_disease', 'work_type', 'smoking_status', 'is_urban',
    'is_male', 'age_group', 'bmi_category', 'avg_glucose_category', 'ever_smoked',
    'ever_married', 'work_residence_interaction',
]

df = pd.get_dummies(df, columns=cat_encode_cols, drop_first=True)

for col in df.columns:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(np.int8)

In [11]:
df.head(3)

,age,avg_glucose_level,bmi,stroke,age_bmi_interaction,health_risk_score,marital_age_interaction,hypertension_1,heart_disease_1,work_type_Never_worked,...,ever_married_1,work_residence_interaction_Govt_job_Urban,work_residence_interaction_Never_worked_Rural,work_residence_interaction_Never_worked_Urban,work_residence_interaction_Private_Rural,work_residence_interaction_Private_Urban,work_residence_interaction_Self-employed_Rural,work_residence_interaction_Self-employed_Urban,work_residence_interaction_children_Rural,work_residence_interaction_children_Urban
0,67.0,228.69,36.6,1,2452.2,2.67,67.0,0,1,0,...,1,0,0,0,0,1,0,0,0,0
2,80.0,105.92,32.5,1,2600.0,1.80,80.0,0,1,0,...,1,0,0,0,1,0,0,0,0,0
3,49.0,171.23,34.4,1,1685.6,1.49,49.0,0,0,0,...,1,0,0,0,0,1,0,0,0,0


In [12]:
df.dtypes

age                                               float64
avg_glucose_level                                 float64
bmi                                               float64
stroke                                              int64
age_bmi_interaction                               float64
health_risk_score                                 float64
marital_age_interaction                           float64
hypertension_1                                       int8
heart_disease_1                                      int8
work_type_Never_worked                               int8
work_type_Private                                    int8
work_type_Self-employed                              int8
work_type_children                                   int8
smoking_status_formerly smoked                       int8
smoking_status_never smoked                          int8
smoking_status_smokes                                int8
is_urban_1                                           int8
is_male_1     

In [13]:
y = df["stroke"]
X = df.drop("stroke", axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

In [14]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

num_features = [
    'age', 'avg_glucose_level', 'bmi', 'age_bmi_interaction',
    'health_risk_score', 'marital_age_interaction'
]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_features] = scaler.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler.transform(X_test[num_features])

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

X_train_scaled.head()

,age,avg_glucose_level,bmi,age_bmi_interaction,health_risk_score,marital_age_interaction,hypertension_1,heart_disease_1,work_type_Never_worked,work_type_Private,...,ever_married_1,work_residence_interaction_Govt_job_Urban,work_residence_interaction_Never_worked_Rural,work_residence_interaction_Never_worked_Urban,work_residence_interaction_Private_Rural,work_residence_interaction_Private_Urban,work_residence_interaction_Self-employed_Rural,work_residence_interaction_Self-employed_Urban,work_residence_interaction_children_Rural,work_residence_interaction_children_Urban
4649,0.096680,0.298815,0.155914,0.032788,0.020737,0.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1172,0.487305,0.293125,0.548387,0.315078,0.366359,0.00000,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
49,0.475098,0.013967,0.776882,0.392327,0.625576,0.47561,1,0,0,1,...,1,0,0,0,1,0,0,0,0,0
2346,0.169922,0.438535,0.333333,0.081326,0.036447,0.00000,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
1134,0.018066,0.181622,0.263441,0.007763,0.003875,0.00000,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [15]:
X_train_scaled.dtypes

age                                               float64
avg_glucose_level                                 float64
bmi                                               float64
age_bmi_interaction                               float64
health_risk_score                                 float64
marital_age_interaction                           float64
hypertension_1                                       int8
heart_disease_1                                      int8
work_type_Never_worked                               int8
work_type_Private                                    int8
work_type_Self-employed                              int8
work_type_children                                   int8
smoking_status_formerly smoked                       int8
smoking_status_never smoked                          int8
smoking_status_smokes                                int8
is_urban_1                                           int8
is_male_1                                            int8
age_group_18-3